# Shadow Revenue Detection System

# Dashboard Queries

## Objective

This notebook prepares the final datasets required for dashboard visualization.

The dashboard will consume these queries directly from the Gold layer.

In [0]:
CATALOG = "shadow_revenue_catalog"
GOLD = "gold"

In [0]:
# Load gold tables
fact_bad = spark.table(f"{CATALOG}.{GOLD}.fact_revenue_bad")
fact_good = spark.table(f"{CATALOG}.{GOLD}.fact_revenue_good")
kpi = spark.table(f"{CATALOG}.{GOLD}.kpi_comparison")

## Dashboard 1

Display the overall business KPIs for both pipelines.

In [0]:
%sql
SELECT *
FROM shadow_revenue_catalog.gold.kpi_comparison;

pipeline,total_revenue,total_payment,total_revenue_difference,accuracy_ratio,missing_payments,orphan_payments,price_mismatches,duplicate_orders,revenue_leakages
Good,6.437988398E7,1.906700396E7,4.531288002E7,29.62,2000,600,20000,0,16266
Bad,6.573054318000072E7,1.9450610200000014E7,4.627993297999992E7,29.59,2037,600,20400,400,16595


## Dashboard 2

Compare total revenue and payment between the Bad and Good pipelines.

In [0]:
%sql
SELECT
pipeline,
total_revenue,
total_payment
FROM shadow_revenue_catalog.gold.kpi_comparison;

pipeline,total_revenue,total_payment
Good,6.437988398E7,1.906700396E7
Bad,6.573054318000072E7,1.9450610200000014E7


Databricks visualization. Run in Databricks to view.

## Dashboard 3: Missing Payments

Display the number of orders that do not have a corresponding payment record. This helps identify unpaid orders and potential revenue leakage.

**Recommended Visualization:** Bar Chart

In [0]:
%sql
SELECT
pipeline,
missing_payments
FROM shadow_revenue_catalog.gold.kpi_comparison;

pipeline,missing_payments
Good,2000
Bad,2037


Databricks visualization. Run in Databricks to view.

## Dashboard 4: Orphan Payments

Display the number of payments that do not match any order. These payments cannot be linked to a valid transaction.

**Recommended Visualization:** KPI Card

In [0]:
%sql
SELECT
pipeline,
orphan_payments
FROM shadow_revenue_catalog.gold.kpi_comparison;

pipeline,orphan_payments
Good,600
Bad,600


Databricks visualization. Run in Databricks to view.

## Dashboard 5: Issue Distribution

Show the distribution of major data quality issues detected during processing. This helps identify which anomaly contributes most to revenue leakage.

**Recommended Visualization:** Donut Chart / Pie Chart

In [0]:
%sql
SELECT
'Missing Payments' AS issue,
SUM(missing_payment) AS total
FROM shadow_revenue_catalog.gold.fact_revenue_good

UNION ALL

SELECT
'Price Mismatch',
SUM(price_mismatch)
FROM shadow_revenue_catalog.gold.fact_revenue_good

UNION ALL

SELECT
'Duplicate Orders',
(
SELECT COUNT(*)
FROM shadow_revenue_catalog.silver.orders_bad
)
-
(
SELECT COUNT(*)
FROM shadow_revenue_catalog.silver.orders_good
);

issue,total
Missing Payments,2000
Price Mismatch,20000
Duplicate Orders,400


Databricks visualization. Run in Databricks to view.

## Dashboard 6: Revenue Trend

Track revenue and payment trends over time to identify seasonal patterns, fluctuations, or unexpected changes in financial performance.

**Recommended Visualization:** Line Chart

In [0]:
%sql
SELECT
order_date,
SUM(calculated_revenue) AS revenue,
SUM(payment_amount) AS payment
FROM shadow_revenue_catalog.gold.fact_revenue_good
GROUP BY order_date
ORDER BY order_date;

order_date,revenue,payment
2024-01-01,1220445.5800,335922.0600
2024-01-02,1127475.8800,320810.9600
2024-01-03,1083167.7200,303734.2600
2024-01-04,937500.1200,275845.1000
2024-01-05,1061770.3500,338753.4400
2024-01-06,1216220.8100,319763.0300
2024-01-07,974554.7700,304220.1800
2024-01-08,1260185.7500,336965.8500
2024-01-09,1035037.5200,307963.5000
2024-01-10,958037.2300,304200.0900


Databricks visualization. Run in Databricks to view.

## Dashboard 7: Revenue Difference by Order

Compare expected revenue with the actual payment received for each order. This helps identify transactions contributing to financial discrepancies.

**Recommended Visualization:** Table

In [0]:
%sql
SELECT

order_id,

customer_id,

calculated_revenue,

payment_amount,

revenue_difference

FROM shadow_revenue_catalog.gold.fact_revenue_good

ORDER BY ABS(revenue_difference) DESC;

order_id,customer_id,calculated_revenue,payment_amount,revenue_difference
6063,868,11443.1000,null,11443.1000
4046,1384,11443.1000,null,11443.1000
2227,222,11443.1000,null,11443.1000
1966,1804,11385.1500,null,11385.1500
19044,1649,11443.1000,117.4900,11325.6100
15640,1390,11443.1000,153.8000,11289.3000
4187,862,11385.1500,109.4200,11275.7300
17604,1023,11443.1000,212.0500,11231.0500
10545,1176,11385.1500,173.9100,11211.2400
17414,176,11385.1500,200.9800,11184.1700


Databricks visualization. Run in Databricks to view.

## Dashboard 8: Revenue Leakage

Display orders where the expected revenue is greater than the payment received. These orders represent potential financial leakage.

**Recommended Visualization:** Bar Chart

In [0]:
%sql
SELECT

order_id,

revenue_difference

FROM shadow_revenue_catalog.gold.fact_revenue_good

WHERE revenue_difference > 0

ORDER BY revenue_difference DESC;

order_id,revenue_difference
6063,11443.1000
2227,11443.1000
4046,11443.1000
1966,11385.1500
19044,11325.6100
15640,11289.3000
4187,11275.7300
17604,11231.0500
10545,11211.2400
17414,11184.1700


Databricks visualization. Run in Databricks to view.

## Validation

Verify that all Gold layer tables have been created successfully before building the dashboard.

In [0]:
%sql
SHOW TABLES IN shadow_revenue_catalog.gold

database,tableName,isTemporary
gold,fact_revenue_bad,false
gold,fact_revenue_good,false
gold,kpi_comparison,false
gold,kpi_revenue_bad,false
gold,kpi_revenue_good,false
